In [0]:
%sql
USE CATALOG hive_metastore;

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
SELECT cast(key as String), cast(value as String)
from bronze
limit 20

In [0]:
%sql
SELECT v.*
from (
   select from_json(cast(value as String),"order_id string,order_timestamp timestamp,customer_id string, quantity bigint,total bigint, books array<struct<book_id string,quantity bigint,subtotal bigint>>") as v
   FROM bronze where topic = "orders"
)

In [0]:
(spark.readStream
      .table("bronze")
      .createOrReplaceTempView("bronze_tmp"))

In [0]:
%sql
SELECT v.*
FROM (
  SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
   FROM bronze_tmp
   WHERE topic = "orders")

In [0]:
%sql
 CREATE OR REPLACE TEMPORARY VIEW orders_silver_tmp AS
   SELECT v.*
   FROM (
     SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
     FROM bronze_tmp
     WHERE topic = "orders")

In [0]:
query = (spark.table("orders_silver_tmp")
               .writeStream
               .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/orders_silver")
               .trigger(availableNow=True)
               .table("orders_silver"))

query.awaitTermination()

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table("bronze")
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
        .select("v.*")
     .writeStream
        .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/orders_silver")
        .trigger(availableNow=True)
        .table("orders_silver"))

query.awaitTermination()